# Tree search: BFS (Breadth-First Search), DFS (Depth-First Search), iterative deepening

_Artificial Intelligence (CS221)_

**Explore possibilities one by one. Go wide, go deep, or do a clever mix.**

To find a path, we explore states one at a time. The states branch out like a tree.

     Breadth-first search (BFS) explores level by level: all nearby states first.

---

This notebook builds tree search from scratch, one piece at a time. We'll explore a tiny tree with both BFS and DFS using the *same* loop — the only difference is whether the frontier behaves like a queue or a stack — then watch how that one choice changes the order in which a city map gets visited. Run each cell, read the note above it, and try predicting the visit order before you run it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

### Step 1 — Define the tree and one shared search routine

We explore a small binary tree: node `1` is the root, its children are `2` and `3`, and so on down to the leaves. The key insight is that BFS and DFS are the **same algorithm** — pull a node off the frontier, record it, push its children — and differ only in *which end* of the frontier we pull from. Pulling from the **front** (a queue) gives breadth-first; pulling from the **back** (a stack) gives depth-first. The `use_stack` flag selects between the two.

In [ ]:
children = {1: [2, 3], 2: [4, 5], 3: [6, 7], 4: [], 5: [], 6: [], 7: []}

def search(use_stack):
    frontier = [1]      # start with just the root
    order = []          # the order in which we visit nodes

    while frontier:
        # Stack (DFS): pop from the back. Queue (BFS): pop from the front.
        if use_stack:
            u = frontier.pop()
        else:
            u = frontier.pop(0)

        order.append(u)

        kids = children[u]
        if use_stack:
            kids = reversed(kids)   # so the left child comes off the stack first

        for v in kids:
            frontier.append(v)

    return order

### Step 2 — Compare BFS and DFS visit orders

Now run the same routine both ways. **BFS** (queue) visits level by level: the root, then both of its children, then all four grandchildren — `1, 2, 3, 4, 5, 6, 7`. **DFS** (stack) dives down one branch before backing up: root, left child, left-left leaf, and so on — `1, 2, 4, 5, 3, 6, 7`. Same tree, same code, completely different exploration shape.

In [ ]:
print("BFS (queue):", search(use_stack=False))
print("DFS (stack):", search(use_stack=True))

## Visualize the data & results

_Exploring the SF district map from the Ferry Building, how do BFS and DFS differ in visit order?_

### Step 1 — Build the city map as a graph

We reuse the San Francisco district map: districts are nodes and roads are two-way edges. For each edge `(u, v)` we list `v` as a neighbour of `u` and vice versa, giving the neighbour map `nbr`. Unlike the clean binary tree, this graph has loops, so our search routines will need to track which nodes they've already seen.

In [ ]:
from collections import deque
import matplotlib.pyplot as plt

# Each pair is a two-way road between two districts.
edges = [
    ("FB", "US"), ("FB", "CT"), ("CT", "NB"), ("US", "CT"), ("US", "SO"),
    ("SO", "MS"), ("US", "CC"), ("CC", "MS"), ("MS", "CA"), ("CC", "CA"), ("NB", "US"),
]

# Build the neighbour map. Roads are undirected, so add both directions.
nbr = {}
for u, v in edges:
    nbr.setdefault(u, []).append(v)
    nbr.setdefault(v, []).append(u)

### Step 2 — Implement BFS and DFS on the map

Here are the two searches written out separately so the data structures are obvious. **BFS** uses a `deque` and pops from the *front*, expanding the closest districts first. **DFS** uses a plain list as a *stack* and pops from the *back*, plunging deep into one direction before backtracking. Both record the visit order in `seen`, and both skip nodes they've already visited so the graph's loops don't cause trouble.

In [ ]:
def bfs(start):
    seen = [start]
    fr = deque([start])
    while fr:
        u = fr.popleft()                 # queue: take from the front
        for v in sorted(nbr[u]):
            if v not in seen:
                seen.append(v)
                fr.append(v)
    return seen

def dfs(start):
    seen = []
    st = [start]
    while st:
        u = st.pop()                     # stack: take from the back
        if u in seen:
            continue
        seen.append(u)
        for v in sorted(nbr[u], reverse=True):
            if v not in seen:
                st.append(v)
    return seen

### Step 3 — Plot the visit step of each district

We run both searches from the Ferry Building (`FB`) and, for each district, look up *when* it was visited (its 1-based position in the visit order). Plotting these side by side shows the signature difference: BFS visit steps grow gently with distance from the start, while DFS produces an uneven pattern because it commits to one branch all the way down before turning to the next.

In [ ]:
ids = ["FB", "CT", "US", "NB", "CC", "SO", "CA", "MS"]

b = bfs("FB")
d = dfs("FB")

# For each district, its 1-based position in the visit order.
bstep = [b.index(n) + 1 for n in ids]
dstep = [d.index(n) + 1 for n in ids]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(ids, bstep, color="#4ea1ff")
ax1.set_title("BFS visit step")

ax2.bar(ids, dstep, color="#ffb454")
ax2.set_title("DFS visit step")

plt.tight_layout()
plt.show()